In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm.notebook import tqdm
from TCN_TDE import TCN, tcn_loss,create_gaussian_label

In [2]:

# ================= 1. 设置相对路径并创建目录 =================
BASE_DIR = './tcn_delay_results'
os.makedirs(BASE_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(BASE_DIR, 'tcn_delay_model.pth')
LOSS_PLOT_PATH = os.path.join(BASE_DIR, 'loss_curve.png')

# ================= 2. 数据集构建与软件时延模拟 =================
class DelayEstimationDataset(Dataset):
    def __init__(self, num_samples, seq_length, max_delay, mode='train'):
        self.num_samples = num_samples
        self.seq_length = seq_length
        self.max_delay = max_delay
        self.mode = mode

        # 预生成所有数据，避免训练时动态生成导致的IO瓶颈
        self.signals = np.random.randn(num_samples, seq_length).astype(np.float32)
        self.delays = np.random.randint(-max_delay, max_delay + 1, size=num_samples)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        signal = self.signals[idx]
        delay = self.delays[idx]

        # 软件时延模拟：通过循环移位实现
        delayed_signal = np.roll(signal, delay)

        # 数据维度匹配：将两个通道堆叠为 (seq_length, 2)
        input_data = np.stack([signal, delayed_signal], axis=-1)

        return torch.tensor(input_data, dtype=torch.float32), torch.tensor(delay, dtype=torch.float32)



In [3]:

# ================= 4. 训练配置与执行 =================
# 超参数设置
SEQ_LENGTH = 1000
MAX_DELAY = 50
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-3

# 划分数据集 (80% 训练集, 20% 测试集)
train_dataset = DelayEstimationDataset(num_samples=8000, seq_length=SEQ_LENGTH, max_delay=MAX_DELAY)
test_dataset = DelayEstimationDataset(num_samples=2000, seq_length=SEQ_LENGTH, max_delay=MAX_DELAY)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 初始化模型、优化器
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TCN(input_dim=2,enc_dim=64,bottle_neck_dim=16,hidden_dim=64,
            layer=6,stack=2,max_delay=MAX_DELAY).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 训练循环 (带进度条)
train_losses, test_losses = [], []
print(f"Training on: {device}")

for epoch in range(EPOCHS):
    # --- 训练阶段 ---
    model.train()
    running_train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=True)
    for batch_x, batch_y in train_pbar:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = tcn_loss(outputs, batch_y, MAX_DELAY)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()
        train_pbar.set_postfix({'Loss': f"{loss.item():.4f}"})

    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- 测试阶段 ---
    model.eval()
    running_test_loss = 0.0
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Test] ", leave=True)
        for batch_x, batch_y in test_pbar:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = tcn_loss(outputs, batch_y, MAX_DELAY)
            running_test_loss += loss.item()

    avg_test_loss = running_test_loss / len(test_loader)
    test_losses.append(avg_test_loss)

    print(f">> Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}\n")

# ================= 5. 结果保存与可视化 =================
# 1. 保存模型权重到相对路径
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"[Success] Model saved to: {MODEL_SAVE_PATH}")

# 2. 绘制并保存 Loss 曲线
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(test_losses, label='Test Loss', marker='s')
plt.title('TCN Delay Estimation Training Process')
plt.xlabel('Epoch')
plt.ylabel('Composite Loss')
plt.legend()
plt.grid(True)
plt.savefig(LOSS_PLOT_PATH, dpi=150)
plt.show()
print(f"[Success] Loss curve saved to: {LOSS_PLOT_PATH}")

Training on: cpu


Epoch 1/20 [Train]:   0%|          | 0/125 [00:00<?, ?it/s]

KeyboardInterrupt: 